In [1]:

import queue
import random

import carla
import numpy as np
import open3d as o3d


# ============================================================
# Configuration
# ============================================================

CARLA_HOST = "localhost"
CARLA_PORT = 2000
CARLA_TIMEOUT_SECONDS = 10.0

RANDOM_SEED = 42

LIDAR_RANGE_METERS = 50.0
LIDAR_CHANNELS = 32
POINTS_PER_SECOND = 100_000
ROTATION_FREQUENCY_HZ = 10.0

VEHICLE_FILTER = "vehicle.*"

# Open3D uses a right-handed coordinate system.
# Convert CARLA's Y axis for a more intuitive 3D display.
COORDINATE_TRANSFORM = np.array(
    [1.0, -1.0, 1.0],
    dtype=np.float32,
)


# ============================================================
# CARLA Connection
# ============================================================

def connect_to_carla() -> tuple[carla.Client, carla.World]:
    client = carla.Client(
        CARLA_HOST,
        CARLA_PORT,
    )
    client.set_timeout(CARLA_TIMEOUT_SECONDS)

    return client, client.get_world()


# ============================================================
# Vehicle
# ============================================================

def get_vehicle_blueprint(
    blueprint_library: carla.BlueprintLibrary,
) -> carla.ActorBlueprint:
    blueprints = list(
        blueprint_library.filter(VEHICLE_FILTER)
    )

    if not blueprints:
        raise RuntimeError("No vehicle blueprints available.")

    four_wheeled = [
        blueprint
        for blueprint in blueprints
        if blueprint.has_attribute("number_of_wheels")
        and blueprint.get_attribute(
            "number_of_wheels"
        ).as_int() == 4
    ]

    return random.choice(four_wheeled or blueprints)


def spawn_vehicle(
    world: carla.World,
    blueprint_library: carla.BlueprintLibrary,
) -> carla.Vehicle:
    blueprint = get_vehicle_blueprint(
        blueprint_library
    )

    spawn_points = list(
        world.get_map().get_spawn_points()
    )
    random.shuffle(spawn_points)

    for spawn_point in spawn_points:
        vehicle = world.try_spawn_actor(
            blueprint,
            spawn_point,
        )

        if vehicle is not None:
            return vehicle

    raise RuntimeError(
        "Could not spawn a vehicle. "
        "Try clearing existing actors or changing maps."
    )


# ============================================================
# LiDAR
# ============================================================

def spawn_lidar(
    world: carla.World,
    vehicle: carla.Vehicle,
) -> carla.Sensor:
    blueprint_library = world.get_blueprint_library()

    lidar_blueprint = blueprint_library.find(
        "sensor.lidar.ray_cast"
    )

    lidar_blueprint.set_attribute(
        "range",
        str(LIDAR_RANGE_METERS),
    )
    lidar_blueprint.set_attribute(
        "channels",
        str(LIDAR_CHANNELS),
    )
    lidar_blueprint.set_attribute(
        "points_per_second",
        str(POINTS_PER_SECOND),
    )
    lidar_blueprint.set_attribute(
        "rotation_frequency",
        str(ROTATION_FREQUENCY_HZ),
    )
    lidar_blueprint.set_attribute(
        "upper_fov",
        "10.0",
    )
    lidar_blueprint.set_attribute(
        "lower_fov",
        "-30.0",
    )

    # Mount the sensor above the vehicle roof.
    lidar_transform = carla.Transform(
        carla.Location(
            x=0.0,
            y=0.0,
            z=2.2,
        )
    )

    return world.spawn_actor(
        lidar_blueprint,
        lidar_transform,
        attach_to=vehicle,
    )


# ============================================================
# Sensor Callback
# ============================================================

def lidar_callback(
    measurement: carla.LidarMeasurement,
    data_queue: queue.Queue,
) -> None:
    """Convert the CARLA measurement to an XYZ point cloud."""

    points = np.frombuffer(
        measurement.raw_data,
        dtype=np.float32,
    ).reshape(-1, 4)

    # Copy XYZ values because the callback's data is transient.
    xyz = points[:, :3].copy()

    # Convert to the viewer's coordinate convention.
    xyz *= COORDINATE_TRANSFORM

    # Keep only the latest measurement if the viewer falls behind.
    try:
        data_queue.put_nowait(
            (measurement.frame, xyz)
        )
    except queue.Full:
        try:
            data_queue.get_nowait()
        except queue.Empty:
            pass

        try:
            data_queue.put_nowait(
                (measurement.frame, xyz)
            )
        except queue.Full:
            pass


# ============================================================
# Live Visualization
# ============================================================

def run_viewer(
    lidar: carla.Sensor,
    data_queue: queue.Queue,
) -> None:
    viewer = o3d.visualization.Visualizer()

    viewer.create_window(
        window_name="LearnCARLA - Live LiDAR",
        width=1280,
        height=720,
    )

    point_cloud = o3d.geometry.PointCloud()
    geometry_added = False

    # Draw the vehicle/sensor origin.
    axes = o3d.geometry.TriangleMesh.create_coordinate_frame(
        size=2.0,
        origin=[0.0, 0.0, 0.0],
    )
    viewer.add_geometry(axes)

    render_options = viewer.get_render_option()
    render_options.background_color = np.array(
        [0.05, 0.05, 0.05]
    )
    render_options.point_size = 2.0

    print("Live LiDAR viewer running.")
    print("Rotate: left mouse | Zoom: scroll")
    print("Close the viewer or press Ctrl+C to stop.")

    try:
        while viewer.poll_events():
            try:
                frame, xyz = data_queue.get_nowait()
            except queue.Empty:
                viewer.update_renderer()
                continue

            if xyz.size == 0:
                continue

            point_cloud.points = o3d.utility.Vector3dVector(
                xyz
            )

            # Color points by distance from the sensor.
            distances = np.linalg.norm(
                xyz,
                axis=1,
            )

            normalized = np.clip(
                distances / LIDAR_RANGE_METERS,
                0.0,
                1.0,
            )

            colors = np.column_stack(
                (
                    normalized,
                    1.0 - normalized,
                    np.full_like(normalized, 0.2),
                )
            )

            point_cloud.colors = (
                o3d.utility.Vector3dVector(colors)
            )

            if not geometry_added:
                viewer.add_geometry(point_cloud)
                geometry_added = True
            else:
                viewer.update_geometry(point_cloud)

            viewer.update_renderer()

            print(
                f"\rFrame: {frame} | "
                f"Points: {len(xyz):,}",
                end="",
                flush=True,
            )

    finally:
        lidar.stop()
        viewer.destroy_window()


# ============================================================
# Main
# ============================================================

def main() -> None:
    random.seed(RANDOM_SEED)

    client, world = connect_to_carla()
    blueprint_library = world.get_blueprint_library()

    vehicle = None
    lidar = None

    # Keep the newest measurement; discard stale frames.
    data_queue = queue.Queue(maxsize=1)

    try:
        vehicle = spawn_vehicle(
            world,
            blueprint_library,
        )

        lidar = spawn_lidar(
            world,
            vehicle,
        )

        lidar.listen(
            lambda measurement: lidar_callback(
                measurement,
                data_queue,
            )
        )

        print("=== LIVE LIDAR ===")
        print(f"Vehicle ID: {vehicle.id}")
        print(f"Vehicle type: {vehicle.type_id}")
        print(f"LiDAR range: {LIDAR_RANGE_METERS} m")
        print(f"LiDAR channels: {LIDAR_CHANNELS}")

        run_viewer(
            lidar,
            data_queue,
        )

    except KeyboardInterrupt:
        print("\nStopping LiDAR viewer...")

    finally:
        if lidar is not None and lidar.is_alive:
            lidar.stop()
            lidar.destroy()

        if vehicle is not None and vehicle.is_alive:
            vehicle.destroy()

        print("\nCARLA actors cleaned up.")


if __name__ == "__main__":
    main()


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
=== LIVE LIDAR ===
Vehicle ID: 33
Vehicle type: vehicle.carlamotors.firetruck
LiDAR range: 50.0 m
LiDAR channels: 32
Live LiDAR viewer running.
Rotate: left mouse | Zoom: scroll
Close the viewer or press Ctrl+C to stop.
Frame: 104087 | Points: 69243
CARLA actors cleaned up.
